# Toxic gas leak dispersion: run everything on a GPU

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → **Run all** (about 20 minutes).

Part A is the 2D advection–diffusion core from the synopsis: serial, OpenMP, MPI and three CUDA kernels, all checked against each other.
Part B is the 3D Vizag 2020 simulator (Lattice Boltzmann air flow + gas transport over real terrain).
Part C opens both live dashboards.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os
REPO = 'https://github.com/swastigupta8/gas-leak-pcap.git'  # if the repo is private, upload a zip instead
if not os.path.exists('/content/gas-leak-pcap'):
    !git clone -q $REPO /content/gas-leak-pcap
%cd /content/gas-leak-pcap
!git log -1 --oneline

In [ ]:
!apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null 2>&1
!pip -q install pytest fastapi "uvicorn[standard]" websockets

## A1. Build the 2D core (CPU and GPU)

In [ ]:
!make -C core2d -j4 > /dev/null && make -C core2d cuda CUDA_ARCH=sm_75 && ls core2d/build

## A2. Correctness
The test suite checks that OpenMP, MPI (1–4 ranks, blocking and overlapped) and the emulated GPU kernels are bit-identical to the serial code, and that the physics matches the analytic solution.

In [ ]:
!python -m pytest tests -q

Now the real GPU. The GPU may fuse multiply–add operations, so it is compared with a tolerance rather than bit for bit.

In [ ]:
import json, subprocess, numpy as np
B = 'core2d/build'
args = ['--nx', '1000', '--ny', '700', '--steps', '400', '--source', '300,450,5000', '--u', '2.5', '--v', '-1.2', '--D', '4']
def run(exe, extra, out):
    r = subprocess.run([f'{B}/{exe}', *extra, *args, '--out', out], capture_output=True, text=True, check=True)
    return json.loads(r.stdout.strip().splitlines()[-1]), np.fromfile(out, np.float32)
_, ref = run('gas2d', ['--variant', 'serial'], '/tmp/ref.bin')
for k in ['naive', 'shared', 'coarse']:
    info, f = run('gas2d_cuda', ['--variant', k], f'/tmp/{k}.bin')
    rel = np.abs(f - ref).max() / ref.max()
    print(f"{k:7s} max difference {rel:.2e} of peak  ->  {'PASS' if rel < 1e-4 else 'FAIL'}")

## A3. Benchmarks on the 4000 × 4000 grid
Throughput of every implementation, OpenMP and MPI scaling, and each GPU kernel's achieved memory bandwidth against the T4's peak.

In [ ]:
!python scripts/benchmark.py
from IPython.display import Image, display
for f in ['variants', 'openmp_scaling', 'mpi', 'gpu_bandwidth']:
    display(Image(f'results/{f}.png', width=640))

In [ ]:
import json
R = json.load(open('results/bench.json'))
print(f"{'variant':10s} {'ms/step':>9s} {'MLUPS':>9s} {'x real time':>12s} {'speed-up':>9s}")
base = R['variants'][0]['ms_per_step']
for r in R['variants']:
    print(f"{r['variant']:10s} {r['ms_per_step']:9.3f} {r['mlups']:9.0f} {r['realtime_factor']:12.1f} {base / r['ms_per_step']:9.1f}")

## B1. Build the 3D simulator

In [ ]:
%cd /content/gas-leak-pcap/sim3d
!mkdir -p results site
!make -j4 > /dev/null && make cuda CUDA_ARCH=native && ls build

## B2. Correctness: GPU vs CPU (3D)

In [ ]:
!cd build && ./bench_cuda check && ./bench_cpu check
import numpy as np
a = np.fromfile('build/check_cuda.bin', np.float32); b = np.fromfile('build/check_cpu.bin', np.float32)
n = len(a) // 2
dC = np.abs(a[:n] - b[:n]).max(); dU = np.abs(a[n:] - b[n:]).max()
print(f"gas: max difference {dC:.3g} ppm (peak {b[:n].max():.0f} ppm), wind: max difference {dU:.3g} lattice units")
print("PASS" if dC < 1e-2 * b[:n].max() and dU < 1e-3 else "CHECK THIS: GPU and CPU disagree")

## B3. Real site data
Terrain (Copernicus 90 m), buildings and place names (OpenStreetMap), and the hourly weather for 6–7 May 2020 (ERA5). Takes 1–3 minutes.

In [ ]:
!python data/prepare_site.py --size 5000 --dx 10
import json, numpy as np, matplotlib.pyplot as plt
d = np.load('site/vizag.npz'); meta = json.loads(str(d['meta']))
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(d['terrain'], origin='lower', cmap='terrain'); ax[0].set_title('Terrain height (m above lowest point)')
ax[1].imshow(d['buildings'] > 0, origin='lower', cmap='Greys'); ax[1].set_title('Buildings (OpenStreetMap)')
for a in ax:
    a.plot(meta['plant']['x'], meta['plant']['y'], 'r^', ms=10)
    for p in meta['places'][:25]:
        a.plot(p['x'], p['y'], 'o', color='orange' if p['reported'] else 'k', ms=4)
        a.text(p['x'] + 3, p['y'] + 3, ('★ ' if p['reported'] else '') + p['name'], fontsize=7)
plt.tight_layout(); plt.show()
print('Weather that night (IST):')
for h in meta['weather'][:8]: print(f"  {h['t'][11:16]}  {h['temp']:.1f} °C  wind {h['wind']:.1f} m/s from {h['dir']:.0f}°  cloud {h['cloud']}%")

## B4. Benchmarks (3D)

In [ ]:
import json, math, subprocess, numpy as np, matplotlib.pyplot as plt
d = np.load('site/vizag.npz')
top = float((d['terrain'] + d['buildings']).max()); dx = float(d['dx'])
NX, NY = d['terrain'].shape[1], d['terrain'].shape[0]
NZ = int(min(64, max(8, math.ceil((top + 80) / dx))))
run = lambda args: json.loads(subprocess.run(args, capture_output=True, text=True, cwd='build').stdout.strip().splitlines()[-1])
print(f'site grid: {NX} x {NY} x {NZ} = {NX*NY*NZ/1e6:.1f} million boxes')
gpu = run(['./bench_cuda', str(NX), str(NY), str(NZ), '100'])
cpu1 = run(['./bench_cpu', str(NX), str(NY), str(NZ), '2', '1'])
cpuN = run(['./bench_cpu', str(NX), str(NY), str(NZ), '2'])
for r in (cpu1, cpuN, gpu): print(r)
res = dict(site_grid=[NX, NY, NZ], gpu=gpu, cpu_serial=cpu1, cpu_openmp=cpuN,
           cpu_air_ms=cpu1['air_ms'], cpu_gas_ms=cpu1['gas_naive_ms'])
# scaling with grid size
sizes = [64, 128, 256, 384]
scal = []
for n in sizes:
    g = run(['./bench_cuda', str(n), str(n), '24', '100']); c = run(['./bench_cpu', str(n), str(n), '24', '3'])
    scal.append(dict(n=n, gpu=g, cpu=c)); print(n, 'GPU', g['air_ms'] + g['gas_shared_ms'], 'ms  CPU', c['air_ms'] + c['gas_naive_ms'], 'ms')
res['scaling'] = scal
json.dump(res, open('results/bench.json', 'w'), indent=1)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
lab = ['CPU 1 core', f"CPU {cpuN['threads']} cores", 'GPU naive', 'GPU shared']
air = [cpu1['air_ms'], cpuN['air_ms'], gpu['air_ms'], gpu['air_ms']]
gas = [cpu1['gas_naive_ms'], cpuN['gas_naive_ms'], gpu['gas_naive_ms'], gpu['gas_shared_ms']]
ax[0].bar(lab, air, label='air (LBM)'); ax[0].bar(lab, gas, bottom=air, label='gas'); ax[0].set_yscale('log')
ax[0].set_ylabel('ms per time step (log scale)'); ax[0].set_title('Time per step on the Vizag grid'); ax[0].legend()
sp = [(cpu1['air_ms'] + cpu1['gas_naive_ms']) / (a + g) for a, g in zip(air, gas)]
ax[1].bar(lab, sp, color=['#888', '#aaa', '#e98', '#e50']); ax[1].set_title('Speedup vs CPU 1 core')
for i, v in enumerate(sp): ax[1].text(i, v, f'{v:.0f}×', ha='center', va='bottom')
ax[2].plot(sizes, [s['cpu']['air_mlups'] for s in scal], 'o-', label='CPU (all cores)')
ax[2].plot(sizes, [s['gpu']['air_mlups'] for s in scal], 'o-', label='GPU')
ax[2].set_xlabel('grid width (boxes), 24 tall'); ax[2].set_ylabel('million boxes updated per second'); ax[2].set_yscale('log'); ax[2].legend(); ax[2].set_title('Throughput vs grid size')
plt.tight_layout(); plt.savefig('results/benchmarks.png', dpi=140); plt.show()
if gpu['gas_shared_ms'] > 0:
    print(f"Gas kernel: shared memory is {gpu['gas_naive_ms'] / gpu['gas_shared_ms']:.2f}× faster than naive on the GPU")

## B5. MPI (3D)
The map is split into strips, one per process; the result must match a single run (`pass: true`).

In [ ]:
import subprocess, json, matplotlib.pyplot as plt
rows = []
for p in [1, 2, 4]:
    out = subprocess.run(f'mpirun --allow-run-as-root --oversubscribe -np {p} ./mpi_cuda 256 256 24 100',
                         shell=True, capture_output=True, text=True, cwd='build').stdout.strip().splitlines()
    r = json.loads(out[-1]); rows.append(r); print(r)
json.dump(rows, open('results/mpi.json', 'w'), indent=1)
plt.figure(figsize=(6, 4))
x = [str(r['ranks']) for r in rows]
plt.bar(x, [r['compute_ms'] for r in rows], label='compute')
plt.bar(x, [r['comm_ms'] for r in rows], bottom=[r['compute_ms'] for r in rows], label='halo exchange')
plt.xlabel('MPI processes (all sharing one GPU on Colab)'); plt.ylabel('ms per step'); plt.legend(); plt.title('MPI: compute vs communication')
plt.savefig('results/mpi.png', dpi=140); plt.show()

## B6. Replay the night of 7 May 2020

In [ ]:
!cd server && python replay.py --minutes 60
import json, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
r = json.load(open('results/replay.json')); z = np.load('results/replay.npz')
meta = json.loads(str(np.load('site/vizag.npz')['meta']))
fig, ax = plt.subplots(1, 2, figsize=(15, 6.5))
pk = np.maximum(z['peak'], 0.1)
im = ax[0].imshow(pk, origin='lower', norm=LogNorm(0.1, 1e4), cmap='inferno')
ax[0].contour(z['peak'], levels=r['aegl'], colors=['#f5d547', '#ff8a2a', '#ff3b3b'], linewidths=1.2)
ax[0].contour(z['terrain'], levels=10, colors='w', linewidths=0.3, alpha=0.4)
plt.colorbar(im, ax=ax[0], label='peak ppm at breathing height')
ax[0].set_title('Worst concentration during the first hour (AEGL-1/2/3 contours)')
im2 = ax[1].imshow(z['arrive'], origin='lower', cmap='viridis')
plt.colorbar(im2, ax=ax[1], label='minutes after 02:30 when AEGL-1 (20 ppm) was first reached')
ax[1].set_title('When the gas arrived')
for a in ax:
    a.plot(*z['plant'], 'c^', ms=10)
    for p in meta['places']:
        a.plot(p['x'], p['y'], 'o', mfc='none', mec='w' if p['reported'] else '#888', ms=6)
        if p['reported']: a.text(p['x'] + 3, p['y'] + 3, p['name'], color='w', fontsize=7)
plt.tight_layout(); plt.savefig('results/replay.png', dpi=140); plt.show()
print(f"{'Place':28s} {'km':>5s} {'peak ppm':>9s} {'AEGL-1 at':>9s}  reported")
for p in sorted(r['places'], key=lambda p: p['dist_m'] or 0):
    print(f"{p['name'][:28]:28s} {(p['dist_m'] or 0)/1000:5.1f} {p['peak']:9.0f} {p['first_clock'] or '-':>9s}  {'★' if p['reported'] else ''}")

## C. Live dashboards
Starts the 2D dashboard (port 8000) and the 3D dashboard (port 8001) on the GPU. Colab serves them through its own private proxy: the links below work only for you, while this notebook is running.

In [ ]:
import subprocess, time, os
from google.colab.output import eval_js
os.chdir('/content/gas-leak-pcap')
subprocess.run('pkill -f "dashboard/server.py"; pkill -f "server/server.py"', shell=True)
subprocess.Popen('PORT=8000 GRID=4000 python dashboard/server.py > dash2d.log 2>&1', shell=True)
subprocess.Popen('cd sim3d && PORT=8001 python server/server.py > ../dash3d.log 2>&1', shell=True)
time.sleep(12)
print(open('dash2d.log').read()); print(open('dash3d.log').read()[-800:])
for port, name in ((8000, '2D dashboard'), (8001, '3D Vizag dashboard')):
    print(f'>>> {name}:', eval_js(f'google.colab.kernel.proxyPort({port})'))